# Notebook 1: PyTorch Essentials for ML Practitioners

You are likely comfortable with NumPy arrays and scikit-learn's highly abstracted `model.fit()`.

Deep learning requires more granular control. In this notebook, we will unpack the "magic" of model training by covering the three pillars of PyTorch:

1. **Tensors:** PyTorch's version of NumPy arrays (with GPU acceleration).
2. **Autograd:** Automatic differentiation to calculate gradients for us.
3. **The Training Loop:** Writing the exact steps that happen inside a standard `fit()` function.

## 1. Tensors: The Foundation

A Tensor is simply an n-dimensional array. If you know NumPy, you basically know Tensors. The crucial difference is that Tensors can track computational history and be moved to GPUs for massive parallel processing.

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. bridging with NumPy
np_array = np.array([[1, 2], [3, 4]])
tensor_from_np = torch.from_numpy(np_array)

print("NumPy Array:\n", np_array)
print("\nPyTorch Tensor:\n", tensor_from_np)

# 2. Moving computations to the GPU (if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

# Move tensor to the device
tensor_gpu = tensor_from_np.to(device)

NumPy Array:
 [[1 2]
 [3 4]]

PyTorch Tensor:
 tensor([[1, 2],
        [3, 4]])

Using device: cuda


## 2. Autograd: Automatic Differentiation

In traditional ML, algorithms like Random Forests use information gain, while OLS regression uses closed-form math. Deep neural networks use Gradient Descent, which requires calculating partial derivatives for millions of parameters.

PyTorch's `autograd` engine dynamically builds a computational graph as you perform operations. When you call `.backward()`, it traverses this graph in reverse to calculate gradients automatically. Let's see it in action.


In [3]:
# Create a tensor and tell PyTorch to track its gradients
x = torch.tensor(3.0, requires_grad=True)

# Perform a simple operation: y = x^2 + 2x + 1
y = x**2 + 2*x + 1
print(f"Result of forward pass (y): {y.item()}")

# Calculate gradients (dy/dx)
y.backward()

# Since y = x^2 + 2x + 1, dy/dx = 2x + 2. 
# At x = 3, dy/dx = 2(3) + 2 = 8.
print(f"Gradient calculated by autograd (dy/dx): {x.grad.item()}")

Result of forward pass (y): 16.0
Gradient calculated by autograd (dy/dx): 8.0


## 3. Building Linear Regression from Scratch

To understand PyTorch, we are going to build a simple Linear Regression model without using any pre-built neural network layers. We will define our weights, our bias, and our forward pass $y = XW + b$ manually.

In [4]:
# Set seed for reproducibility
torch.manual_seed(42)

# Create synthetic data (similar to sklearn.datasets.make_regression)
X = torch.rand(100, 1) * 10  # 100 samples, 1 feature
true_weight = 2.5
true_bias = 4.2
y = true_weight * X + true_bias + torch.randn(100, 1) # Add some noise

# Initialize our model parameters randomly
# requires_grad=True because these are what we want our model to learn/update
W = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f"Initial Random Weight: {W.item():.4f}")
print(f"Initial Random Bias: {b.item():.4f}")

# Define the forward pass function
def forward(X):
    return X @ W + b  # Matrix multiplication + broadcasted bias

# Define the Mean Squared Error (MSE) loss function
def mse_loss(y_pred, y_true):
    return torch.mean((y_pred - y_true)**2)

Initial Random Weight: -1.2842
Initial Random Bias: -0.6917


## 4. The PyTorch Training Loop (Deconstructing `model.fit()`)

This is the most critical concept to master. Every deep learning training loop in PyTorch, whether it's a simple regression model or a massive Large Language Model, follows these exact five steps:

1. **Forward Pass:** Pass data through the model to get predictions.
2. **Calculate Loss:** Compare predictions to the true labels.
3. **Zero Gradients:** Clear old gradients from the previous step (PyTorch accumulates them by default).
4. **Backward Pass:** Compute the gradients of the loss with respect to parameters (`loss.backward()`).
5. **Update Weights:** Adjust the parameters using gradient descent.

In [6]:
learning_rate = 0.01
epochs = 2000

# Track loss for plotting
loss_history = []

for epoch in range(epochs):
    # 1. Forward pass
    y_pred = forward(X)
    
    # 2. Calculate Loss
    loss = mse_loss(y_pred, y)
    loss_history.append(loss.item())
    
    # 3. Backward pass (calculate gradients)
    loss.backward()
    
    # 4. Update Weights (using torch.no_grad() so we don't track this update step in autograd)
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad
        
    # 5. Zero the gradients for the next iteration
    W.grad.zero_()
    b.grad.zero_()
    
    # Print progress
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("\n--- Training Complete ---")
print(f"Learned Weight: {W.item():.4f} (True: {true_weight})")
print(f"Learned Bias: {b.item():.4f} (True: {true_bias})")

Epoch 40/2000 | Loss: 1.1583
Epoch 80/2000 | Loss: 1.0007
Epoch 120/2000 | Loss: 0.8933
Epoch 160/2000 | Loss: 0.8201
Epoch 200/2000 | Loss: 0.7701
Epoch 240/2000 | Loss: 0.7361
Epoch 280/2000 | Loss: 0.7129
Epoch 320/2000 | Loss: 0.6971
Epoch 360/2000 | Loss: 0.6863
Epoch 400/2000 | Loss: 0.6789
Epoch 440/2000 | Loss: 0.6739
Epoch 480/2000 | Loss: 0.6705
Epoch 520/2000 | Loss: 0.6682
Epoch 560/2000 | Loss: 0.6666
Epoch 600/2000 | Loss: 0.6655
Epoch 640/2000 | Loss: 0.6648
Epoch 680/2000 | Loss: 0.6643
Epoch 720/2000 | Loss: 0.6639
Epoch 760/2000 | Loss: 0.6637
Epoch 800/2000 | Loss: 0.6635
Epoch 840/2000 | Loss: 0.6634
Epoch 880/2000 | Loss: 0.6633
Epoch 920/2000 | Loss: 0.6633
Epoch 960/2000 | Loss: 0.6633
Epoch 1000/2000 | Loss: 0.6632
Epoch 1040/2000 | Loss: 0.6632
Epoch 1080/2000 | Loss: 0.6632
Epoch 1120/2000 | Loss: 0.6632
Epoch 1160/2000 | Loss: 0.6632
Epoch 1200/2000 | Loss: 0.6632
Epoch 1240/2000 | Loss: 0.6632
Epoch 1280/2000 | Loss: 0.6632
Epoch 1320/2000 | Loss: 0.6632
Epo